# 06 — Comparar CPC gerado pelo modelo base e pelo adaptador

Este notebook compara **20 famílias de validação**, com uma resposta do Qwen3-8B base e outra do adaptador selecionado no notebook 05: **40 gerações**. Não executa treinamento nem abre a partição de teste.

As duas variantes recebem o mesmo título/resumo pt/en e a mesma instrução CPC, sem o código de referência na entrada do modelo. A geração é determinística, com até 32 novos tokens e contexto total de até 1024. A pontuação compara o código completo com o CPC proxy congelado.

**Antes de executar:** atualize no Drive este notebook, `scripts/avaliar_cpc_geracao.py`, `config/perfil_avaliacao_cpc_validacao.json` e `SHA256SUMS.txt`. Preserve os auxiliares `scripts/testar_qlora_gpu.py` e `scripts/treinar_qlora_cpc.py` já usados no 05, os dados, os dois relatórios do 05 e a pasta completa do checkpoint selecionado. Abra o **06** no Colab, escolha **GPU** e execute as células em ordem.

A validação já foi usada para selecionar a época 3; esta comparação é um diagnóstico de desenvolvimento. A estimativa final será feita depois, nas 20 famílias de teste, com o procedimento fixado. Acerto CPC não comprova qualidade dos seis campos da extração. O alvo é um proxy alfanumérico da família, não uma classificação principal oficial.

## 1. Montar o Drive e selecionar a pasta

Mantenha a mesma pasta do projeto. Os caminhos dos relatórios do treinamento e do checkpoint escolhido já estão no perfil desta avaliação. Os pesos grandes ficam no cache temporário `/content`; os resultados novos ficam no Drive. Não carregue E5 ou outro modelo simultaneamente.

In [ ]:
from pathlib import Path
from google.colab import drive
import os

os.environ['HF_HOME'] = '/content/hf_cache_llm_eidt_br'
os.environ['USE_TF'] = '0'
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
os.environ['PYTHONUNBUFFERED'] = '1'
drive.mount('/content/drive')
PASTA_DRIVE = '/content/drive/MyDrive/Replicação LLM EIDT IC' # @param {type:'string'}

raiz = Path(PASTA_DRIVE).expanduser().resolve(strict=True)
ponto_de_montagem = Path('/content/drive').resolve(strict=True)
if raiz == ponto_de_montagem or not raiz.is_relative_to(ponto_de_montagem) or not raiz.is_dir():
    raise ValueError('Selecione a pasta do projeto dentro do Drive montado.')

def caminho_no_projeto(relativo):
    parte = Path(relativo)
    if not relativo or parte.is_absolute() or '..' in parte.parts:
        raise ValueError(f'O caminho deve ser relativo ao projeto: {relativo!r}')
    alvo = (raiz / parte).resolve()
    if alvo == raiz or not alvo.is_relative_to(raiz):
        raise ValueError('O caminho informado sai da pasta do projeto.')
    return alvo

perfil_relativo = 'config/perfil_avaliacao_cpc_validacao.json'
script_relativo = 'scripts/avaliar_cpc_geracao.py'
print('Pasta selecionada:', raiz)
print('Avaliação: 20 famílias de validação, 2 variantes, nenhum novo treinamento.')

## 2. Conferir arquivos, relatórios e checkpoint

Esta célula verifica as assinaturas dos programas, configurações, validação e relatórios do 05. Em seguida, o programa confere o manifesto e os arquivos do checkpoint antes de qualquer download ou carregamento do modelo. A conferência não carrega o estado do otimizador nem abre dados de teste.

Se faltar algum arquivo, copie-o para a subpasta indicada. Não altere os hashes para contornar a conferência e não substitua o relatório completo pelo resumo copiado no chat.

In [ ]:
from datetime import datetime, timezone
from hashlib import sha256
from importlib import metadata
import json, re, subprocess, sys, tempfile

SHA256_PERFIL_AVALIACAO = 'ac58a8fee416bdc2925950b2adae8908758e3561dbe08c6d53528ea2948c51ef'
SHA256_SCRIPT_AVALIACAO = '16681c351cd2481fdae7c1d40f938ab31ee59a1e73b446242940c21a9fe19b4b'
ID_PERFIL = 'br_patstat_200_qwen3_8b_cpc_validacao_v1'
assinaturas = {}
for linha in (raiz / 'SHA256SUMS.txt').read_text(encoding='utf-8-sig').splitlines():
    if not linha.strip():
        continue
    item = re.fullmatch(r'([0-9a-f]{64})  (.+)', linha)
    if not item:
        raise ValueError('Há uma linha inválida no índice SHA256SUMS.txt.')
    digest, nome = item.groups()
    if nome in assinaturas:
        raise ValueError(f'Entrada duplicada em SHA256SUMS.txt: {nome}')
    assinaturas[nome] = digest

conferidos = {}
def ler_conferido(relativo, esperado):
    if not re.fullmatch(r'[0-9a-f]{64}', str(esperado)):
        raise ValueError(f'Assinatura ausente ou inválida: {relativo}')
    arquivo = caminho_no_projeto(relativo)
    if not arquivo.is_file():
        raise FileNotFoundError(f'Copie este arquivo para o Drive: {relativo}')
    conteudo = arquivo.read_bytes()
    digest = sha256(conteudo).hexdigest()
    if digest != esperado:
        raise ValueError(f'Arquivo de outra versão ou alterado: {relativo}. Atualize o conjunto do 06.')
    conferidos[relativo] = digest
    return conteudo

for nome in ('requirements.txt', perfil_relativo, script_relativo,
             'scripts/testar_qlora_gpu.py', 'scripts/treinar_qlora_cpc.py'):
    ler_conferido(nome, assinaturas.get(nome))
    print('Conferido:', nome)
perfil = json.loads(ler_conferido(perfil_relativo, SHA256_PERFIL_AVALIACAO))
ler_conferido(script_relativo, SHA256_SCRIPT_AVALIACAO)
if perfil['id'] != ID_PERFIL:
    raise ValueError('Use o perfil de avaliação CPC publicado para o notebook 06.')
if perfil['avaliacao']['particao'] != 'validacao' or perfil['avaliacao']['familias'] != 20:
    raise ValueError('O notebook 06 aceita somente as 20 famílias de validação.')
esperado_geracao = {'max_prompt_tokens': 992, 'max_new_tokens': 32, 'contexto_total': 1024,
                    'do_sample': False, 'num_beams': 1, 'use_cache': True, 'enable_thinking': False}
if perfil['geracao'] != esperado_geracao:
    raise ValueError('Os parâmetros de geração diferem do protocolo publicado.')
origem = perfil['origem']
for chave in ('config', 'perfil_treino', 'auditoria_preprocessamento', 'validacao',
              'relatorio_treino', 'preparacao_treino', 'helper_teste', 'helper_treino'):
    ler_conferido(origem[chave], origem['sha256_' + chave])

inicio_notebook = datetime.now(timezone.utc)
pasta_relatorios = caminho_no_projeto(perfil['resultados'])
pasta_relatorios.mkdir(parents=True, exist_ok=True)
relatorio_preparacao = pasta_relatorios / f'preparacao_avaliacao_notebook_{inicio_notebook:%Y%m%dT%H%M%S%fZ}.json'
preparacao = {'inicio_utc': inicio_notebook.isoformat(), 'estado': 'arquivos_conferidos',
              'perfil': perfil['id'], 'sha256_arquivos': conferidos,
              'cache_temporario': os.environ['HF_HOME'], 'particao': 'validacao',
              'teste_aberto': False}

def salvar_preparacao():
    preparacao['atualizado_utc'] = datetime.now(timezone.utc).isoformat()
    temporario = relatorio_preparacao.with_suffix('.tmp')
    temporario.write_text(json.dumps(preparacao, ensure_ascii=False, indent=2) + '\n', encoding='utf-8')
    temporario.replace(relatorio_preparacao)

try:
    verificacao = subprocess.run(
        [sys.executable, '-u', str(raiz / script_relativo), '--raiz', str(raiz),
         '--perfil', perfil_relativo, '--validar_apenas'],
        capture_output=True, text=True, check=True,
    )
    integridade = json.loads(verificacao.stdout)
    if integridade.get('estado') != 'integridade_avaliacao_aprovada' or integridade.get('familias') != 20:
        raise RuntimeError('A conferência do checkpoint não retornou a aprovação esperada.')
    preparacao['integridade'] = integridade
    preparacao['estado'] = 'checkpoint_conferido'
    print('Checkpoint conferido:', integridade['checkpoint'])
    print('Famílias: 20 de validação · variantes: base e adaptado · gerações: 40')
    print('Nenhum arquivo de teste aberto. Nenhum modelo carregado nesta célula.')
except Exception as erro:
    preparacao['estado'] = 'integridade_requer_atencao'
    preparacao['tipo_erro'] = type(erro).__name__
    if isinstance(erro, subprocess.CalledProcessError):
        print((erro.stderr or erro.stdout or '')[-4000:])
    raise
finally:
    salvar_preparacao()

arquivo_requisitos = raiz / 'requirements.txt'
print('Preparação:', relatorio_preparacao)

## 3. Instalar as dependências preservando o ambiente do Colab

São as mesmas versões fixadas usadas no notebook 05. A célula preserva PyTorch e NumPy nativos. O processo principal deste notebook não importa PyTorch: a conferência de GPU e a avaliação acontecem em processos separados, depois da instalação.

In [ ]:
if preparacao.get('estado') != 'checkpoint_conferido':
    raise RuntimeError('Conclua a conferência do checkpoint antes de instalar dependências.')
try:
    versoes_protegidas = {nome: metadata.version(nome) for nome in ("torch", "numpy")}
    preparacao["versoes_protegidas"] = versoes_protegidas
    desejadas = {}
    for linha in arquivo_requisitos.read_text(encoding="utf-8-sig").splitlines():
        linha = linha.split("#", 1)[0].strip()
        if linha:
            if not re.fullmatch(r"[A-Za-z0-9_.-]+==[^\s]+", linha):
                raise ValueError("Os requisitos devem usar versões exatas no formato pacote==versão.")
            nome, versao = linha.split("==", 1)
            desejadas[nome] = versao
    divergentes = []
    for nome, esperada in desejadas.items():
        try:
            atual = metadata.version(nome)
        except metadata.PackageNotFoundError:
            atual = None
        if atual != esperada:
            divergentes.append(nome)
    preparacao["pacotes_que_requeriam_instalacao"] = divergentes
    if divergentes:
        with tempfile.TemporaryDirectory(dir="/content") as pasta_temporaria:
            restricoes = Path(pasta_temporaria) / "base_colab.txt"
            plano_pip = Path(pasta_temporaria) / "operacoes_pip.json"
            restricoes.write_text(
                "".join(f"{nome}=={versao}\n" for nome, versao in versoes_protegidas.items()),
                encoding="utf-8",
            )
            subprocess.run(
                [sys.executable, "-m", "pip", "install", "--disable-pip-version-check",
                 "--progress-bar", "off", "-c", str(restricoes), "-r", str(arquivo_requisitos),
                 "--report", str(plano_pip)],
                check=True,
            )
            operacoes = json.loads(plano_pip.read_text(encoding="utf-8"))["install"]
            preparacao["pacotes_alterados"] = sorted(item["metadata"]["name"] for item in operacoes)
    else:
        preparacao["pacotes_alterados"] = []
    for nome, anterior in versoes_protegidas.items():
        if metadata.version(nome) != anterior:
            raise RuntimeError(f"A versão protegida de {nome} mudou. Interrompa e confira o ambiente.")
    preparacao["versoes_requisitos"] = {nome: metadata.version(nome) for nome in desejadas}
    if preparacao["versoes_requisitos"] != desejadas:
        raise RuntimeError("Alguma dependência ainda diverge da versão fixada.")
    preparacao["estado"] = "dependencias_conferidas"
    print("Dependências conferidas. PyTorch e NumPy nativos preservados.")
except Exception as erro:
    preparacao["estado"] = "falha_na_instalacao"
    preparacao["tipo_erro"] = type(erro).__name__
    raise
finally:
    salvar_preparacao()

## 4. Conferir GPU, RAM e armazenamento

O carregamento baixa aproximadamente 16 GB de pesos originais quando eles ainda não estão em cache. Com o cache completo, a célula exige 4 GiB livres; sem ele, 22 GiB. A base quantizada será carregada uma vez e o adaptador será ativado ou desativado para a comparação. Não serão mantidas duas cópias do Qwen3-8B na GPU.

In [ ]:
if preparacao.get('estado') != 'dependencias_conferidas':
    raise RuntimeError('Conclua a instalação e conferência das dependências antes de continuar.')
codigo_recursos = r"""
import json, os, platform, shutil
from pathlib import Path
import torch
ram = {}
for linha in Path('/proc/meminfo').read_text().splitlines():
    chave, _, valor = linha.partition(':')
    if chave in ('MemTotal', 'MemAvailable'):
        ram[chave] = round(int(valor.split()[0]) / 1024**2, 2)
saida = {'python': platform.python_version(), 'torch': str(torch.__version__),
         'cuda_pytorch': torch.version.cuda, 'cuda_disponivel': torch.cuda.is_available(),
         'disco_livre_gib': round(shutil.disk_usage('/content').free / 1024**3, 2), 'ram_gib': ram}
if saida['cuda_disponivel']:
    livre, total = torch.cuda.mem_get_info(0)
    saida.update(gpu=torch.cuda.get_device_name(0), capacidade_cuda=list(torch.cuda.get_device_capability(0)),
                 vram_livre_gib=round(livre/1024**3, 3), vram_total_gib=round(total/1024**3, 3))
print(json.dumps(saida))
"""
try:
    verificador = subprocess.run([sys.executable, '-c', codigo_recursos], capture_output=True, text=True, check=True)
    recursos = json.loads(verificador.stdout.strip().splitlines()[-1])
    snapshot = (Path(os.environ['HF_HOME']) / 'hub' / 'models--Qwen--Qwen3-8B' /
                'snapshots' / perfil['modelo']['revisao'])
    indice_pesos = snapshot / 'model.safetensors.index.json'
    cache_completo = False
    if indice_pesos.is_file():
        fragmentos = set(json.loads(indice_pesos.read_text())['weight_map'].values())
        cache_completo = bool(fragmentos) and all(
            Path(nome).name == nome and (snapshot / nome).is_file() and (snapshot / nome).stat().st_size > 0
            for nome in fragmentos
        )
    recursos['cache_pesos_completo'] = cache_completo
    recursos['disco_minimo_gib'] = 4 if cache_completo else 22
    preparacao['recursos'] = recursos
    print(json.dumps(recursos, ensure_ascii=False, indent=2))
    if not recursos['cuda_disponivel']:
        raise RuntimeError('Selecione GPU no tipo de ambiente de execução do Colab.')
    if recursos['capacidade_cuda'][0] < 6:
        raise RuntimeError('A GPU não atende à capacidade CUDA mínima para este perfil NF4.')
    if recursos['disco_livre_gib'] < recursos['disco_minimo_gib']:
        raise RuntimeError('O armazenamento local está abaixo da reserva necessária. Revise o espaço antes da avaliação.')
    preparacao['estado'] = 'pronto_para_avaliacao'
    print('Ambiente preparado. A próxima célula inicia as 40 gerações de avaliação.')
except Exception as erro:
    preparacao['estado'] = 'recursos_requerem_atencao'
    preparacao['tipo_erro'] = type(erro).__name__
    if isinstance(erro, subprocess.CalledProcessError):
        print((erro.stderr or '')[-4000:])
    raise
finally:
    salvar_preparacao()

## 5. Gerar e comparar os códigos CPC

O programa recarrega o adaptador do passo 30 e gera respostas para as duas variantes, sem disponibilizar o alvo ao modelo. Não há amostragem, busca pelo melhor resultado ou correção automática de respostas. Apenas normalização NFC, maiúsculas e remoção de espaços são aplicadas na comparação.

A métrica principal é o acerto exato do código integral, com término por EOS. Uma resposta com prosa, códigos adicionais ou interrompida pelo limite não é contada como acerto. O indicador de formato descreve somente a estrutura sintática; não consulta o catálogo oficial CPC. Respostas brutas, tokens, falhas de formato e pares de acertos/erros ficam registrados.

Se a avaliação parar, preserve o relatório e as previsões parciais. Nenhuma métrica final será apresentada como completa antes das 40 gerações. Uma repetição cria outra pasta de execução e conserva a anterior; não retreina nem escolhe outro adaptador.

In [ ]:
if preparacao.get('estado') != 'pronto_para_avaliacao':
    raise RuntimeError('Execute primeiro as conferências de arquivos, dependências e recursos.')
chamada = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
saida_relativa = f"{perfil['resultados']}/chamada_avaliacao_{chamada}.json"
relatorio_chamada = caminho_no_projeto(saida_relativa)
comando = [sys.executable, '-u', str(raiz / script_relativo),
           '--raiz', str(raiz), '--perfil', perfil_relativo, '--saida_relatorio', saida_relativa]
preparacao['relatorio_avaliacao'] = saida_relativa
preparacao['estado'] = 'avaliacao_em_execucao'
salvar_preparacao()
codigo_saida = None
interrompido = False
relatorio = None
print('Relatório desta chamada:', relatorio_chamada, flush=True)
try:
    processo = subprocess.run(comando, check=False)
    codigo_saida = processo.returncode
except KeyboardInterrupt:
    interrompido = True
    print('Avaliação interrompida. Preserve o relatório e as previsões parciais.')
finally:
    if relatorio_chamada.is_file():
        try:
            relatorio = json.loads(relatorio_chamada.read_text(encoding='utf-8'))
            campos = ('estado', 'familias', 'geracoes_concluidas', 'familias_por_variante',
                      'metricas', 'comparacao_pareada', 'pico_alocado_gib', 'pico_reservado_gib',
                      'duracao_segundos', 'arquivo_predicoes', 'arquivo_csv', 'erro')
            resumo = {nome: relatorio.get(nome) for nome in campos}
            print('\nRESUMO DA AVALIAÇÃO — copie este bloco')
            print(json.dumps(resumo, ensure_ascii=False, indent=2))
        except (ValueError, OSError) as erro_leitura:
            print('Não foi possível ler o relatório:', str(erro_leitura))
    else:
        print('O programa não chegou a criar o relatório. Preserve a mensagem de erro acima.')
    concluido = codigo_saida == 0 and relatorio and relatorio.get('estado') == 'avaliacao_cpc_concluida'
    preparacao['estado'] = 'avaliacao_concluida' if concluido else 'avaliacao_requer_atencao'
    preparacao['codigo_saida'] = codigo_saida
    preparacao['interrompido'] = interrompido
    salvar_preparacao()
    print('Relatório completo:', relatorio_chamada)
    print('Preparação do ambiente:', relatorio_preparacao)
if interrompido:
    raise RuntimeError('Avaliação interrompida. Confira o relatório antes de repetir.')
if not concluido:
    raise RuntimeError('A avaliação não foi concluída. Confira o resumo e o erro acima.')
print('\nAvaliação concluída. Relatório, respostas e comparação estão no Drive.')

## 6. O que enviar e o que vem depois

Envie o bloco **RESUMO DA AVALIAÇÃO**, o JSON indicado em **Relatório completo** e o CSV indicado em `arquivo_csv`. O arquivo de preparação do ambiente é útil se houver uma falha. Preserve também o JSONL de respostas brutas e a pasta completa do treinamento 05.

O relatório permitirá verificar se o adaptador melhora o acerto do CPC inteiro e quais famílias mudam de resultado. Uma perda menor no treinamento não garante esse ganho. Com apenas 20 famílias de validação e 17 códigos integrais ausentes no treino, os números são exploratórios; cada acerto representa 5 pontos percentuais.

A sequência prevista é: revisar este diagnóstico e fixar o procedimento; avaliar uma vez nas 20 famílias de teste; comparar a extração dos seis grupos do artigo com evidências textuais e revisão independente; gerar embeddings E5 e fazer o alinhamento; calcular os indicadores possíveis e documentar os dados que ainda faltam. Esses passos posteriores não são executados pelo notebook 06.

Se não houver mais trabalho nesta sessão, desconecte e exclua o ambiente para encerrar o uso da GPU. Os resultados gravados no Drive permanecem; o cache temporário de pesos desaparece.